# Student Budget Assistant
**ISYS2001 – Assessment 2: Programming Project**

A small finance assistant that helps an international student in Perth see where their money goes each month, check their spending against a budget they set, plan a savings goal, and ask questions about their own spending in plain English.

**How to run:** add your Gemini key as a Colab Secret called `GEMINI_API_KEY` (key icon on the left), then **Runtime → Run all**. The app appears under the `app.launch()` cell and the tests run at the end.

**Notebook map**
1. Step 1 – Understand the problem
2. Step 2 – Inputs and outputs
3. Step 3 – Worked example by hand
4. Step 4 – Pseudocode
5. Step 5 – Python code (setup → loading data → summary → custom tools → Gemini → Gradio app)
6. Step 6 – Testing (normal cases, edge and error cases, bugs found and fixed)

## Step 1 – Understand the problem

**User.** An international university student living in Perth. They pay rent weekly, work a casual job, and spend on groceries, transport, eating out and subscriptions. Money comes in weekly but goes out in many small purchases, so by the end of the month they often don't know where it went.

**Business problem.** Small, frequent purchases (food delivery, subscriptions) add up without the student noticing. A banking app shows a list of transactions, but it doesn't say *"you are 40% over your eating-out budget"* or *"at this rate you will reach your $1,200 flight-home goal in 12 weeks"*. The student needs something that turns their own transaction list into decisions.

**What the assistant does**
- Reads the student's own transaction CSV.
- Summarises income, spending and spending by category.
- **Budget checker (custom tool 1):** compares spending in each category with limits the student sets and marks each one *Under*, *Close to limit* or *Over*.
- **Savings goal planner (custom tool 2):** works out how much to save each week to reach a goal by a deadline, and whether the student's current surplus is enough.
- **Chat assistant:** "Penny", a Gemini-powered budget coach that answers questions using the student's real numbers, not generic advice.

**Out of scope.** No investment, tax or credit advice, and no connection to a real bank.

**Success looks like:** the student uploads their CSV and within a minute can answer *"Where did my money go, which category is out of control, and how much should I put aside each week?"*

## Step 2 – Inputs and outputs

| Part | Inputs | Outputs |
|---|---|---|
| Load data | A CSV file with columns `date, description, category, amount` | A list of transaction dictionaries, plus a list of warnings about rows that were skipped or fixed |
| Summary | The list of transactions | A dictionary: income, spending, net (income − spending), spending per category, biggest category, number of transactions |
| **Budget checker** | Spending per category + budget limits typed as text, e.g. `Groceries=250, Eating Out=100` | For each budgeted category: spent, limit, remaining, % used, status (`Under`, `Close to limit` at 80% or more, `Over`) |
| **Savings planner** | Goal amount, amount already saved, number of weeks, weekly surplus | Amount still needed, saving needed per week, whether the surplus is enough, how many weeks it would take at the current rate |
| Chat assistant | The question + a text description of the user's data + earlier messages | A reply from Gemini in the "Penny" persona |

**Rules I decided on**
- Rows with the category `Income` are income; every other category is spending.
- A negative amount in a spending category is a refund, so it reduces that category.
- Amounts may contain `$` or commas (`$1,200.50`); these are removed before converting to a number.
- Rows with a missing or unreadable amount are skipped and reported as a warning, not silently dropped.
- A blank category becomes `Uncategorised`. Categories are changed to Title Case so `groceries` and `Groceries` count as the same category.
- Budget limits must be numbers greater than 0. For savings: goal > 0, saved ≥ 0, weeks must be a whole number above 0.

## Step 3 – Worked example by hand

A tiny made-up month with five rows:

| date | description | category | amount |
|---|---|---|---|
| 01/09 | Casual job pay | Income | 500.00 |
| 02/09 | Coles | Groceries | 60.00 |
| 03/09 | Uber Eats | Eating Out | 45.00 |
| 05/09 | Aldi | Groceries | 40.00 |
| 06/09 | Uber Eats | Eating Out | 70.00 |

**Summary by hand**
- Income = **500.00**
- Groceries = 60 + 40 = **100.00**, Eating Out = 45 + 70 = **115.00**
- Spending = 100 + 115 = **215.00**
- Net = 500 − 215 = **285.00**
- Biggest category = **Eating Out** (115.00)

**Budget check by hand**, limits `Groceries=150, Eating Out=100`
- Groceries: 100 ÷ 150 × 100 = 66.7% used, remaining 50.00 → **Under**
- Eating Out: 115 ÷ 100 × 100 = 115% used, remaining −15.00 → **Over**
- If Groceries had been 125: 125 ÷ 150 × 100 = 83.3%, which is 80% or more → **Close to limit**

**Savings plan by hand**: goal $1,200 (flight home), already saved $300, 10 weeks, surplus $80 a week
- Still needed = 1200 − 300 = **900**
- Needed per week = 900 ÷ 10 = **90.00**
- 80 is less than 90 → **not on track**, short by **10.00** a week
- Weeks at the current rate = 900 ÷ 80 = 11.25 → round **up** to **12 weeks** (after 11 weeks I would only have $880)

These numbers are the expected results in my tests in Step 6.

## Step 4 – Pseudocode

```
FUNCTION clean_amount(text)
    remove spaces, "$" and "," from the text
    IF the text is now empty: raise an error "missing amount"
    RETURN the text converted to a number (float raises an error for text like "abc")

FUNCTION clean_category(text)
    remove spaces at the start and end
    IF empty: RETURN "Uncategorised"
    RETURN the text in Title Case

FUNCTION load_transactions(file)
    read the CSV with pandas (raise a friendly error if the file is missing or empty)
    replace empty cells with ""
    tidy the column names; IF a required column is missing: raise an error
    transactions = empty list, warnings = empty list
    FOR each row:
        TRY to clean the amount
        IF that fails: add a warning, skip to the next row
        clean the category (add a warning if it was blank)
        add a dictionary {date, description, category, amount} to transactions
    IF no transactions were kept: raise an error
    RETURN transactions, warnings

FUNCTION summarise(transactions)
    income = 0, by_category = empty dictionary
    FOR each transaction:
        IF category is "Income": add amount to income
        ELSE: add amount to by_category[category] (start it at 0 the first time)
    spending = total of all by_category values
    biggest = the category with the largest total
    RETURN a dictionary with income, spending, net, by_category, biggest, count

FUNCTION parse_budgets(text)                  e.g. "Groceries=250, Eating Out=100"
    FOR each piece between commas:
        skip empty pieces
        split the piece at "="; IF it doesn't give exactly 2 parts: raise an error
        name = cleaned category, limit = cleaned amount (error if not a number or not > 0)
        budgets[name] = limit
    IF budgets is empty: raise an error
    RETURN budgets

FUNCTION check_budget(spending_by_category, budgets)       CUSTOM TOOL 1
    FOR each category in budgets:
        spent = spending for that category (0 if none)
        remaining = limit − spent
        percent_used = spent ÷ limit × 100
        IF spent > limit: status = "Over"
        ELSE IF percent_used ≥ 80: status = "Close to limit"
        ELSE: status = "Under"
        add a result dictionary to the results list
    RETURN results

FUNCTION plan_savings(goal, saved, weeks, weekly_surplus)   CUSTOM TOOL 2
    convert all four inputs to numbers (error if any is not a number)
    check goal > 0, saved ≥ 0, weeks is a whole number > 0
    needed = goal − saved; IF needed ≤ 0: RETURN "already reached"
    per_week = needed ÷ weeks
    on_track = weekly_surplus ≥ per_week
    IF weekly_surplus > 0: weeks_at_current_rate = needed ÷ weekly_surplus rounded UP
    ELSE: weeks_at_current_rate = None (never at this rate)
    RETURN a dictionary of the results

FUNCTION ask_gemini(question, data_text, history, api_key)
    IF no api_key: RETURN a message explaining how to add one
    build the request: system instruction + earlier messages + (data_text + question)
    TRY to POST the request to Gemini
    IF it fails or the status is not 200: RETURN a friendly error message
    RETURN the reply text found inside the JSON response

GRADIO APP (4 tabs)
    Upload  → load_transactions + summarise → show summary, table, warnings
    Budget  → parse_budgets + check_budget → show table
    Savings → plan_savings → show message
    Chat    → ask_gemini with a description of the uploaded data → show reply
    Every tab catches errors and shows a message instead of crashing
```

## Step 5 – Convert to Python

### 5.1 Setup

My Gemini API key is **not** written in this notebook. It is stored in Colab Secrets: click the 🔑 icon on the left, add a secret named `GEMINI_API_KEY`, paste the free key from [Google AI Studio](https://aistudio.google.com/apikey), and turn on *Notebook access*.

In [ ]:
# Gradio is not installed in Colab by default (pandas and requests are).
!pip install -q gradio

In [ ]:
import math   # only used for math.ceil (rounding up)
import os     # only used to make the data folder

import pandas as pd
import requests
import gradio as gr

In [ ]:
# Read the API key from Colab Secrets. If it isn't there (or this isn't Colab),
# the key is None and the chat tab will explain what to do instead of crashing.
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = None

GEMINI_MODEL = "gemini-flash-latest"   # always points to the current free Flash model

if GEMINI_API_KEY:
    print("Gemini API key found.")
else:
    print("No Gemini API key found. The chat tab won't work until you add one; everything else will.")

In [ ]:
# Create the sample CSV files inside Colab, so the notebook runs by itself without
# downloading anything. (The data/ folder in my repository has the same three files.)
os.makedirs("data", exist_ok=True)

SAMPLE_FILES = {
    "data/sample_transactions.csv": """
date,description,category,amount
2026-09-01,Casual job pay - cafe,Income,520.00
2026-09-01,Rent - shared house Bentley,Rent,250.00
2026-09-02,Coles Bentley,Groceries,64.35
2026-09-03,Transperth SmartRider top-up,Transport,30.00
2026-09-04,Guzman y Gomez,Eating Out,18.90
2026-09-05,Spotify Student,Subscriptions,6.99
2026-09-06,Aldi Cannington,Groceries,41.20
2026-09-07,Boost Mobile recharge,Phone,30.00
2026-09-08,Rent - shared house Bentley,Rent,250.00
2026-09-08,Casual job pay - cafe,Income,480.00
2026-09-09,Hungry Jacks,Eating Out,12.50
2026-09-10,Coles Bentley,Groceries,58.10
2026-09-11,Netflix Basic,Subscriptions,10.99
2026-09-12,Uber Eats,Eating Out,34.75
2026-09-13,Kmart Carousel,Shopping,45.00
2026-09-14,Transperth SmartRider top-up,Transport,30.00
2026-09-15,Rent - shared house Bentley,Rent,250.00
2026-09-15,Casual job pay - cafe,Income,505.00
2026-09-16,Bubble tea Northbridge,Eating Out,8.50
2026-09-17,Aldi Cannington,Groceries,49.80
2026-09-18,Curtin campus bookshop,Study,89.95
2026-09-19,Uber Eats,Eating Out,29.40
2026-09-20,Synergy electricity share,Utilities,62.00
2026-09-21,Coles Bentley,Groceries,52.65
2026-09-22,Rent - shared house Bentley,Rent,250.00
2026-09-22,Casual job pay - cafe,Income,495.00
2026-09-23,Event Cinemas Innaloo,Entertainment,22.00
2026-09-24,Transperth SmartRider top-up,Transport,30.00
2026-09-25,Uber Eats,Eating Out,41.20
2026-09-26,Aldi Cannington,Groceries,38.75
2026-09-27,Adobe Creative Cloud student plan,Subscriptions,32.99
2026-09-28,Nando's Carousel,Eating Out,24.60
2026-09-29,Rent - shared house Bentley,Rent,250.00
2026-09-29,Casual job pay - cafe,Income,510.00
2026-09-30,Coles Bentley,Groceries,47.30
""",
    "data/second_student.csv": """
date,description,category,amount
2026-09-01,Parents transfer,Income,1500.00
2026-09-01,Student accommodation,Rent,1200.00
2026-09-03,Woolworths,Groceries,120.00
2026-09-05,Uber Eats,Eating Out,85.00
2026-09-10,JB Hi-Fi,Shopping,299.00
2026-09-12,Uber Eats,Eating Out,60.00
2026-09-15,Transperth,Transport,40.00
2026-09-20,Woolworths,Groceries,95.00
2026-09-25,Gym membership,Health,65.00
""",
    "data/messy_transactions.csv": """
date,description,category,amount
2026-09-01,Casual job pay,income,500
2026-09-02,Coles," groceries ",$60.50
2026-09-03,Mystery charge,,15.00
2026-09-04,Typo row,Eating Out,abc
2026-09-05,Refund Kmart,Shopping,-20.00
2026-09-06,Rent,Rent,
""",
}

for path, text in SAMPLE_FILES.items():
    with open(path, "w") as file:
        file.write(text.strip())
    print("Created", path)

### 5.2 Loading and cleaning the user's data

In [ ]:
REQUIRED_COLUMNS = ["date", "description", "category", "amount"]


def clean_amount(text):
    # Turn text like "$1,200.50" into the number 1200.5. Raises ValueError if it can't.
    text = str(text).strip()
    text = text.replace("$", "")
    text = text.replace(",", "")
    if text == "":
        raise ValueError("missing amount")
    return float(text)   # float() raises ValueError by itself for text like "abc"


def clean_category(text):
    # " groceries " -> "Groceries", and a blank category -> "Uncategorised"
    text = str(text).strip()
    if text == "":
        return "Uncategorised"
    return text.title()


def load_transactions(file_path):
    # Read a CSV file and return (list of transaction dictionaries, list of warnings).
    try:
        df = pd.read_csv(file_path, dtype=str)   # read everything as text; I clean it myself
    except FileNotFoundError:
        raise ValueError("I couldn't find that file.")
    except pd.errors.EmptyDataError:
        raise ValueError("The file is empty.")
    except pd.errors.ParserError:
        raise ValueError("That doesn't look like a valid CSV file.")

    df = df.fillna("")   # empty cells become "" instead of NaN

    # Tidy the column names so " Amount" and "amount" both work
    clean_columns = []
    for column in df.columns:
        clean_columns.append(column.strip().lower())
    df.columns = clean_columns

    for column in REQUIRED_COLUMNS:
        if column not in df.columns:
            raise ValueError(f"The CSV needs a '{column}' column. Expected: date, description, category, amount")

    transactions = []
    warnings = []
    for index, row in df.iterrows():
        row_number = index + 2   # +1 for the header row, +1 because index starts at 0

        try:
            amount = clean_amount(row["amount"])
        except ValueError:
            warnings.append(f"Row {row_number} skipped: '{row['amount']}' is not a valid amount.")
            continue   # go straight to the next row

        category = clean_category(row["category"])
        if category == "Uncategorised":
            warnings.append(f"Row {row_number}: no category, so it was filed under 'Uncategorised'.")

        transaction = {
            "date": row["date"],
            "description": row["description"],
            "category": category,
            "amount": amount,
        }
        transactions.append(transaction)

    if len(transactions) == 0:
        raise ValueError("No usable transactions were found in the file.")
    return transactions, warnings


transactions, warnings = load_transactions("data/sample_transactions.csv")
print(len(transactions), "transactions loaded,", len(warnings), "warnings")
print(transactions[0])

### 5.3 Summarising spending

I used my own dictionary and loop logic here instead of pandas `groupby`, so I can walk through exactly how each total is built. I round to cents each time I add, because computers store decimals slightly inaccurately (0.1 + 0.2 gives 0.30000000000000004).

In [ ]:
def summarise(transactions):
    # Add up income and spending per category from a list of transaction dictionaries.
    income = 0
    by_category = {}

    for transaction in transactions:
        category = transaction["category"]
        amount = transaction["amount"]
        if category == "Income":
            income = round(income + amount, 2)
        else:
            if category not in by_category:
                by_category[category] = 0
            by_category[category] = round(by_category[category] + amount, 2)

    spending = 0
    for total in by_category.values():
        spending = round(spending + total, 2)

    biggest = None
    for category in by_category:
        if biggest is None or by_category[category] > by_category[biggest]:
            biggest = category

    return {
        "income": income,
        "spending": spending,
        "net": round(income - spending, 2),
        "by_category": by_category,
        "biggest_category": biggest,
        "count": len(transactions),
    }


summary = summarise(transactions)
summary

### 5.4 Custom tool 1 – Budget checker

The user types their budgets as one line of text, e.g. `Groceries=300, Eating Out=120`. `parse_budgets` turns that text into a dictionary, and `check_budget` compares it with their real spending.

In [ ]:
CLOSE_TO_LIMIT_PERCENT = 80


def parse_budgets(text):
    # "Groceries=250, Eating Out=100"  ->  {"Groceries": 250.0, "Eating Out": 100.0}
    budgets = {}
    for piece in text.split(","):
        piece = piece.strip()
        if piece == "":
            continue   # allows a comma at the end

        parts = piece.split("=")
        if len(parts) != 2:
            raise ValueError(f"'{piece}' should look like Groceries=250 (and don't put commas inside numbers).")

        name = clean_category(parts[0])
        try:
            limit = clean_amount(parts[1])
        except ValueError:
            raise ValueError(f"The budget for {name} is not a number.")
        if limit <= 0:
            raise ValueError(f"The budget for {name} must be more than 0.")

        budgets[name] = limit

    if len(budgets) == 0:
        raise ValueError("Please enter at least one budget, e.g. Groceries=250")
    return budgets


def check_budget(spending_by_category, budgets):
    # Compare spending with each budget. Returns one result dictionary per budget.
    results = []
    for category in budgets:
        limit = budgets[category]
        spent = spending_by_category.get(category, 0)   # 0 if nothing was spent in this category
        remaining = round(limit - spent, 2)
        percent_used = round(spent / limit * 100, 1)    # safe: parse_budgets makes sure limit > 0

        if spent > limit:
            status = "Over"
        elif percent_used >= CLOSE_TO_LIMIT_PERCENT:
            status = "Close to limit"
        else:
            status = "Under"

        results.append({
            "category": category,
            "spent": spent,
            "limit": limit,
            "remaining": remaining,
            "percent_used": percent_used,
            "status": status,
        })
    return results


budgets = parse_budgets("Groceries=300, Eating Out=120, Subscriptions=20, Transport=100")
pd.DataFrame(check_budget(summary["by_category"], budgets))

### 5.5 Custom tool 2 – Savings goal planner

In [ ]:
def to_number(value, name):
    # Convert what the user typed into a number, with an error message that names the box.
    try:
        return clean_amount(value)
    except ValueError:
        raise ValueError(f"{name} must be a number.")


def plan_savings(goal, saved, weeks, weekly_surplus):
    # Work out what it takes to reach a savings goal. Returns a dictionary.
    goal = to_number(goal, "Goal amount")
    saved = to_number(saved, "Amount already saved")
    weeks = to_number(weeks, "Number of weeks")
    weekly_surplus = to_number(weekly_surplus, "Weekly surplus")

    if goal <= 0:
        raise ValueError("Goal amount must be more than 0.")
    if saved < 0:
        raise ValueError("Amount already saved can't be negative.")
    if weeks <= 0 or weeks != int(weeks):
        raise ValueError("Number of weeks must be a whole number above 0.")

    needed = round(goal - saved, 2)
    if needed <= 0:
        return {"already_reached": True}

    per_week = round(needed / weeks, 2)
    on_track = weekly_surplus >= per_week

    if weekly_surplus > 0:
        weeks_at_current_rate = math.ceil(needed / weekly_surplus)   # always round UP
    else:
        weeks_at_current_rate = None   # with no surplus the goal is never reached

    return {
        "already_reached": False,
        "needed": needed,
        "weeks": int(weeks),
        "per_week": per_week,
        "on_track": on_track,
        "shortfall_per_week": round(max(per_week - weekly_surplus, 0), 2),
        "weeks_at_current_rate": weeks_at_current_rate,
    }


def estimate_weekly_surplus(summary):
    # The sample file covers about one month (30 days), so: net ÷ 30 days × 7 days.
    return round(summary["net"] / 30 * 7, 2)


plan_savings(1200, 300, 10, estimate_weekly_surplus(summary))

### 5.6 Penny, the budget coach (Gemini)

I call the Gemini API with `requests.post`, the same way we called web APIs in class: build a dictionary, send it as JSON, and read the reply out of the JSON that comes back.

**How the answers are grounded in the user's data (R2):** before every question, `describe_data` turns the user's summary into a short block of text, and that text is sent together with the question. The system instruction tells Penny to use only those figures.

The JSON Gemini sends back looks like this, which is why the reply is read with `["candidates"][0]["content"]["parts"][0]["text"]`:
```
{"candidates": [ {"content": {"parts": [ {"text": "Your biggest cost is rent..."} ], "role": "model"}} ]}
```

In [ ]:
SYSTEM_INSTRUCTION = '''You are "Penny", a friendly budget coach for university students in Australia.
Your purpose is to help the student understand and improve their own spending, using ONLY the
figures in the DATA section of each message. All amounts are Australian dollars.

Rules:
- Every number you mention must come from the DATA. Never invent transactions or totals.
- If no data has been uploaded, say so and ask the student to upload their CSV in the first tab.
- Keep answers short (under 150 words), practical and encouraging, with one or two concrete actions.
- You are not a licensed financial adviser. Do not give investment, tax, credit or legal advice;
  suggest the student speaks to Curtin's student support services or a licensed adviser instead.
- If the question is not about personal finance or budgeting, politely say that is outside what
  you help with and bring the conversation back to their budget.
- If the question is unclear, ask one short clarifying question.'''


def describe_data(summary, budget_results):
    # Turn the user's numbers into plain text that the language model can read.
    if summary is None:
        return "DATA: The student has not uploaded any data yet."

    text = "DATA (this month):\n"
    text += f"Income: ${summary['income']:.2f}\n"
    text += f"Spending: ${summary['spending']:.2f}\n"
    text += f"Left over (income - spending): ${summary['net']:.2f}\n"
    text += f"Biggest spending category: {summary['biggest_category']}\n"
    text += "Spending by category:\n"
    for category in summary["by_category"]:
        text += f"  {category}: ${summary['by_category'][category]:.2f}\n"

    if budget_results is not None:
        text += "Budget check:\n"
        for result in budget_results:
            text += f"  {result['category']}: spent ${result['spent']:.2f} of ${result['limit']:.2f} ({result['status']})\n"
    return text


def build_gemini_request(question, data_text, history):
    # history is a list like [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
    # Gemini calls the assistant "model", so I change the role name while copying.
    contents = []
    for message in history:
        if message["role"] == "user":
            gemini_role = "user"
        else:
            gemini_role = "model"
        contents.append({"role": gemini_role, "parts": [{"text": message["content"]}]})

    # The newest question goes last, with the latest data attached to it.
    contents.append({"role": "user", "parts": [{"text": data_text + "\nQUESTION: " + question}]})

    return {
        "system_instruction": {"parts": [{"text": SYSTEM_INSTRUCTION}]},
        "contents": contents,
    }


def ask_gemini(question, data_text, history, api_key):
    # Send the question to Gemini and return its reply, or a friendly message if something goes wrong.
    if not api_key:
        return ("I can't reach the language model yet. Add a GEMINI_API_KEY secret in Colab "
                "(key icon on the left), then run the notebook again.")

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent"
    try:
        response = requests.post(
            url,
            headers={"x-goog-api-key": api_key},   # the key goes in a header, not in the URL
            json=build_gemini_request(question, data_text, history),
            timeout=30,                            # give up after 30 seconds instead of hanging
        )
    except requests.exceptions.RequestException:
        return "I couldn't connect to Gemini. Check your internet connection and try again."

    if response.status_code == 429:
        return "Gemini's free-tier limit was reached. Please wait a minute and try again."
    if response.status_code != 200:
        return f"Gemini returned an error (status {response.status_code}). Check the API key."

    data = response.json()
    try:
        return data["candidates"][0]["content"]["parts"][0]["text"]
    except (KeyError, IndexError):
        return "Gemini sent back an empty answer. Please try asking in a different way."

In [ ]:
# Live check (needs the API key): the answer should mention this student's real figures.
print(ask_gemini("Which category should I cut back on first?", describe_data(summary, None), [], GEMINI_API_KEY))

### 5.7 The Gradio interface

Each tab calls the functions above. Every function that Gradio calls (`on_upload`, `on_budget`, ...) catches `ValueError` and shows the message on screen, so bad input gives an explanation instead of a crash.

**Design choice:** this app runs in Colab for one person at a time, so I keep the uploaded data in one simple dictionary, `app_data`, and the conversation in one list, `chat_history`. A real website with many users at once would need each user's data kept separately (Gradio's `gr.State`), but that would add complexity this project doesn't need.

In [ ]:
app_data = {"summary": None, "budget_results": None}
chat_history = []   # [{"role": "user" or "assistant", "content": "..."}]

STATUS_LABELS = {"Over": "🔴 Over", "Close to limit": "🟠 Close to limit", "Under": "🟢 Under"}


def money(amount):
    # 1234.5 -> "$1,234.50"   and   -464 -> "-$464.00"
    if amount < 0:
        return f"-${-amount:,.2f}"
    return f"${amount:,.2f}"


def on_upload(file_path):
    # Tab 1. Returns: summary text, category table, warnings text, weekly surplus for tab 3.
    if file_path is None:
        return "Please upload a CSV file.", None, "", 0
    try:
        transactions, warnings = load_transactions(file_path)
    except ValueError as error:
        return f"**Problem with the file:** {error}", None, "", 0

    summary = summarise(transactions)
    app_data["summary"] = summary
    app_data["budget_results"] = None   # old budget results belong to the old file

    text = "### Your month at a glance\n"
    text += f"- **Income:** {money(summary['income'])}\n"
    text += f"- **Spending:** {money(summary['spending'])}\n"
    text += f"- **Left over:** {money(summary['net'])}\n"
    text += f"- **Biggest category:** {summary['biggest_category']}\n"
    text += f"- **Transactions read:** {summary['count']}"

    rows = []
    for category in summary["by_category"]:
        spent = summary["by_category"][category]
        if summary["spending"] > 0:
            share = round(spent / summary["spending"] * 100, 1)
        else:
            share = 0   # avoids dividing by zero when the file only has income
        rows.append({"Category": category, "Spent ($)": spent, "% of spending": share})
    table = pd.DataFrame(rows, columns=["Category", "Spent ($)", "% of spending"])
    table = table.sort_values("Spent ($)", ascending=False)

    if len(warnings) == 0:
        warning_text = "No problems found in the file."
    else:
        warning_text = "\n".join(warnings)   # one warning per line

    return text, table, warning_text, estimate_weekly_surplus(summary)


def on_budget(budget_text):
    # Tab 2. Returns: message, budget table.
    if app_data["summary"] is None:
        return "Please upload your transactions in the first tab first.", None
    try:
        budgets = parse_budgets(budget_text)
    except ValueError as error:
        return f"**Problem with your budgets:** {error}", None

    spending = app_data["summary"]["by_category"]
    results = check_budget(spending, budgets)
    app_data["budget_results"] = results   # so Penny can talk about the budget too

    rows = []
    over = []
    for result in results:
        rows.append({
            "Category": result["category"],
            "Spent": money(result["spent"]),
            "Budget": money(result["limit"]),
            "Remaining": money(result["remaining"]),
            "% used": f"{result['percent_used']}%",
            "Status": STATUS_LABELS[result["status"]],
        })
        if result["status"] == "Over":
            over.append(result["category"])

    if len(over) > 0:
        message = "**Over budget:** " + ", ".join(over)
    else:
        message = "**Nice work - nothing is over budget.**"

    for category in budgets:
        if category not in spending:
            message += f"\n\nNo spending found for {category}. Check it is spelled the same as in your CSV."

    return message, pd.DataFrame(rows)


def on_savings(goal, saved, weeks, surplus):
    # Tab 3. Returns: a message.
    try:
        plan = plan_savings(goal, saved, weeks, surplus)
    except ValueError as error:
        return f"**Problem:** {error}"

    if plan["already_reached"]:
        return "🎉 You've already reached this goal!"

    text = f"- You still need **{money(plan['needed'])}**.\n"
    text += f"- To finish in {plan['weeks']} weeks, save **{money(plan['per_week'])} per week**.\n"
    if plan["on_track"]:
        text += f"- ✅ Your surplus covers it. At that rate you'd finish in about **{plan['weeks_at_current_rate']} weeks**."
    elif plan["weeks_at_current_rate"] is None:
        text += "- ❌ You aren't saving anything at the moment, so you'll need to cut spending to reach this goal."
    else:
        text += f"- ❌ You're **{money(plan['shortfall_per_week'])} per week short**. "
        text += f"At your current rate it would take **{plan['weeks_at_current_rate']} weeks**."
    return text


def on_chat(message):
    # Tab 4. Returns: "" (to empty the question box) and the conversation to display.
    if message.strip() == "":
        return "", chat_history
    data_text = describe_data(app_data["summary"], app_data["budget_results"])
    reply = ask_gemini(message, data_text, chat_history, GEMINI_API_KEY)
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": reply})
    return "", chat_history


def on_clear_chat():
    chat_history.clear()
    return []

In [ ]:
with gr.Blocks(title="Student Budget Assistant") as app:
    gr.Markdown("# 💰 Student Budget Assistant\nUpload your transactions, check your budget, plan a goal, and ask Penny, your budget coach.")

    with gr.Tab("1. Upload"):
        gr.Markdown("Your CSV needs the columns `date, description, category, amount`. Use the category `Income` for money coming in.")
        file_input = gr.File(label="Transactions CSV", file_types=[".csv"], type="filepath")
        gr.Examples(
            examples=["data/sample_transactions.csv", "data/second_student.csv", "data/messy_transactions.csv"],
            inputs=file_input,
        )
        summary_output = gr.Markdown()
        category_output = gr.Dataframe(label="Spending by category")
        warnings_output = gr.Textbox(label="File check", lines=3)

    with gr.Tab("2. Budget checker"):
        budget_input = gr.Textbox(label="Your monthly budgets",
                                  value="Groceries=300, Eating Out=120, Subscriptions=20, Transport=100")
        budget_button = gr.Button("Check my budget", variant="primary")
        budget_message = gr.Markdown()
        budget_output = gr.Dataframe(label="Budget check")

    with gr.Tab("3. Savings goal"):
        goal_input = gr.Number(label="Goal amount ($)", value=1200)
        saved_input = gr.Number(label="Already saved ($)", value=300)
        weeks_input = gr.Number(label="Weeks until I need it", value=10)
        surplus_input = gr.Number(label="What I have left over per week ($) - filled in from your upload", value=0)
        savings_button = gr.Button("Plan my savings", variant="primary")
        savings_output = gr.Markdown()

    with gr.Tab("4. Ask Penny"):
        chatbot = gr.Chatbot(label="Penny, your budget coach", height=380)
        question_input = gr.Textbox(label="Your question", placeholder="e.g. Where is most of my money going?")
        send_button = gr.Button("Send", variant="primary")
        clear_button = gr.Button("Clear chat")

    # Connect each button / input to the function it should run
    file_input.change(on_upload, inputs=file_input,
                      outputs=[summary_output, category_output, warnings_output, surplus_input])
    budget_button.click(on_budget, inputs=budget_input, outputs=[budget_message, budget_output])
    savings_button.click(on_savings, inputs=[goal_input, saved_input, weeks_input, surplus_input],
                         outputs=savings_output)
    send_button.click(on_chat, inputs=question_input, outputs=[question_input, chatbot])
    question_input.submit(on_chat, inputs=question_input, outputs=[question_input, chatbot])  # pressing Enter
    clear_button.click(on_clear_chat, inputs=None, outputs=chatbot)

app.launch()

## Step 6 – Testing

The expected values come from the worked example in Step 3, so each test checks a specific number, not just that something came back. If an `assert` fails, Run all stops at that cell and shows which line failed.

The worked example from Step 3, written as data:

In [ ]:
example_transactions = [
    {"date": "2026-09-01", "description": "Casual job pay", "category": "Income",     "amount": 500.00},
    {"date": "2026-09-02", "description": "Coles",          "category": "Groceries",  "amount": 60.00},
    {"date": "2026-09-03", "description": "Uber Eats",      "category": "Eating Out", "amount": 45.00},
    {"date": "2026-09-05", "description": "Aldi",           "category": "Groceries",  "amount": 40.00},
    {"date": "2026-09-06", "description": "Uber Eats",      "category": "Eating Out", "amount": 70.00},
]
example_summary = summarise(example_transactions)

### Normal cases

In [ ]:
# Test 1: the summary matches my hand calculation
assert example_summary["income"] == 500.00
assert example_summary["spending"] == 215.00
assert example_summary["net"] == 285.00
assert example_summary["by_category"] == {"Groceries": 100.00, "Eating Out": 115.00}
assert example_summary["biggest_category"] == "Eating Out"
assert example_summary["count"] == 5
print("✓ Test 1: summary matches the hand calculation")

# Test 2: the budget check matches my hand calculation
results = check_budget(example_summary["by_category"], {"Groceries": 150, "Eating Out": 100})
groceries = results[0]
eating_out = results[1]
assert groceries["status"] == "Under"
assert groceries["percent_used"] == 66.7
assert groceries["remaining"] == 50.00
assert eating_out["status"] == "Over"
assert eating_out["percent_used"] == 115.0
assert eating_out["remaining"] == -15.00
print("✓ Test 2: budget check matches the hand calculation")

# Test 3: 125 of 150 is 83.3%, which should be "Close to limit"
assert check_budget({"Groceries": 125}, {"Groceries": 150})[0]["status"] == "Close to limit"
print("✓ Test 3: close to limit")

# Test 4: the savings plan matches my hand calculation
plan = plan_savings(1200, 300, 10, 80)
assert plan["needed"] == 900.00
assert plan["per_week"] == 90.00
assert plan["on_track"] == False
assert plan["shortfall_per_week"] == 10.00
assert plan["weeks_at_current_rate"] == 12   # 11.25 rounded UP
print("✓ Test 4: savings plan matches the hand calculation")

# Test 5: budget text is turned into a dictionary, with names tidied
assert parse_budgets("Groceries=250, eating out = 100,") == {"Groceries": 250.0, "Eating Out": 100.0}
assert parse_budgets("Rent=$1000") == {"Rent": 1000.0}
print("✓ Test 5: budget text is read correctly")

# Test 6: the full sample file gives the totals I added up by hand
sample_summary = summarise(load_transactions("data/sample_transactions.csv")[0])
assert sample_summary["income"] == 2510.00
assert sample_summary["by_category"]["Rent"] == 1250.00
assert sample_summary["by_category"]["Eating Out"] == 169.85
assert sample_summary["biggest_category"] == "Rent"
print("✓ Test 6: sample file totals are correct")

# Test 7 (R2): a different student's file gives different answers
second_summary = summarise(load_transactions("data/second_student.csv")[0])
assert second_summary["net"] == -464.00   # 1500 income - 1964 spending: this student overspent
assert second_summary["net"] != sample_summary["net"]
assert "Eating Out: $145.00" in describe_data(second_summary, None)   # Penny receives the real numbers
print("✓ Test 7: different data gives different results")

### Edge cases and invalid input

In [ ]:
# Test 8: amounts with $ and commas are cleaned; bad amounts raise ValueError
assert clean_amount("$1,200.50") == 1200.50
assert clean_amount(" 45 ") == 45.0
assert clean_amount("-20") == -20.0
for bad_amount in ["abc", "", None]:
    try:
        clean_amount(bad_amount)
        raised_error = False
    except ValueError:
        raised_error = True
    assert raised_error, f"clean_amount({bad_amount!r}) should have raised ValueError"
print("✓ Test 8: amounts are cleaned, bad amounts are rejected")

# Test 9: the messy file loads without crashing, and bad rows are reported
messy_transactions, messy_warnings = load_transactions("data/messy_transactions.csv")
messy_summary = summarise(messy_transactions)
assert len(messy_transactions) == 4                       # "abc" row and blank-amount row skipped
assert len(messy_warnings) == 3                           # 2 skipped rows + 1 missing category
assert messy_summary["income"] == 500.00                  # "income" was tidied to "Income"
assert messy_summary["by_category"]["Groceries"] == 60.50  # " groceries " and "$60.50" cleaned
assert messy_summary["by_category"]["Uncategorised"] == 15.00
assert messy_summary["by_category"]["Shopping"] == -20.00  # a refund reduces spending
print("✓ Test 9: messy file handled")

# Test 10: bad files give a clear ValueError instead of crashing
with open("data/wrong_columns.csv", "w") as file:
    file.write("when,what,how much\n2026-09-01,Coles,50\n")
with open("data/empty.csv", "w") as file:
    file.write("")
with open("data/only_header.csv", "w") as file:
    file.write("date,description,category,amount\n")
for bad_file in ["data/does_not_exist.csv", "data/wrong_columns.csv", "data/empty.csv", "data/only_header.csv"]:
    try:
        load_transactions(bad_file)
        raised_error = False
    except ValueError:
        raised_error = True
    assert raised_error, f"{bad_file} should have raised ValueError"
print("✓ Test 10: bad files are rejected with a message")

# Test 11: bad budget text is rejected
for bad_text in ["", "Groceries 250", "Groceries=lots", "Groceries=-50", "Groceries=0", "Rent=1,000"]:
    try:
        parse_budgets(bad_text)
        raised_error = False
    except ValueError:
        raised_error = True
    assert raised_error, f"parse_budgets({bad_text!r}) should have raised ValueError"
print("✓ Test 11: bad budget text is rejected")

# Test 12: budget edge cases
result = check_budget({}, {"Travel": 200})[0]               # a category with no spending
assert result["spent"] == 0 and result["status"] == "Under" and result["remaining"] == 200
result = check_budget({"Shopping": 100}, {"Shopping": 100})[0]   # exactly on the limit is not "Over"
assert result["status"] == "Close to limit"
print("✓ Test 12: budget edge cases")

# Test 13: bad savings input is rejected
bad_savings_inputs = [
    [0, 0, 10, 50],        # goal must be more than 0
    [1000, -5, 10, 50],    # saved can't be negative
    [1000, 0, 0, 50],      # 0 weeks would mean dividing by zero
    [1000, 0, 2.5, 50],    # weeks must be a whole number
    ["lots", 0, 10, 50],   # not a number
    [None, 0, 10, 50],     # an empty box in the app gives None
]
for goal, saved, weeks, surplus in bad_savings_inputs:
    try:
        plan_savings(goal, saved, weeks, surplus)
        raised_error = False
    except ValueError:
        raised_error = True
    assert raised_error, f"plan_savings should have rejected goal={goal}, saved={saved}, weeks={weeks}"
print("✓ Test 13: bad savings input is rejected")

# Test 14: savings edge cases
assert plan_savings(500, 600, 4, 50)["already_reached"] == True
plan = plan_savings(1000, 0, 10, 0)                    # no surplus at all
assert plan["weeks_at_current_rate"] is None and plan["on_track"] == False
assert plan_savings(1000, 0, 10, 100)["on_track"] == True   # exactly enough counts as on track
print("✓ Test 14: savings edge cases")

# Test 15: the chat request really contains the user's data, and a missing key doesn't crash
request = build_gemini_request("How am I doing?", describe_data(example_summary, None), [])
sent_text = request["contents"][-1]["parts"][0]["text"]
assert "Eating Out: $115.00" in sent_text
assert "How am I doing?" in sent_text
assert "Penny" in request["system_instruction"]["parts"][0]["text"]
assert "GEMINI_API_KEY" in ask_gemini("hi", "DATA", [], None)
history = [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "Hello!"}]
request = build_gemini_request("next question", "DATA", history)
assert request["contents"][1]["role"] == "model"       # "assistant" is renamed for Gemini
print("✓ Test 15: chat request contains the user's data")

# Test 16: the app's functions show a message instead of crashing
with open("data/only_income.csv", "w") as file:
    file.write("date,description,category,amount\n2026-09-01,Pay,Income,500\n")
assert "Please upload" in on_upload(None)[0]
assert "Income" in on_upload("data/only_income.csv")[0]   # no spending: must not divide by zero
assert "Problem" in on_upload("data/wrong_columns.csv")[0]
assert "Problem" in on_savings(None, 0, 10, 50)
assert money(-464) == "-$464.00" and money(1234.5) == "$1,234.50"
print("✓ Test 16: the interface handles bad input")

# Test 16 loaded a test file into the app, so clear it again before the app is used
app_data["summary"] = None
app_data["budget_results"] = None

print("\nAll tests passed.")

### Bugs found and fixed

**1. "Run all" failed in Colab with `ValueError: I couldn't find that file.`**
*What happened:* my first version downloaded the sample CSVs from my GitHub repository using a link that still contained the placeholder `YOUR-USERNAME`. GitHub answered with a 404 error, so the `data` folder in Colab stayed empty and `load_transactions("data/sample_transactions.csv")` couldn't find the file.
*How I found it:* the error appeared in section 5.2, but the real cause was in the setup cell, whose output showed `404 Client Error`.
*Fix:* the notebook now creates the three sample files itself in the setup cell, so it doesn't depend on a download, a username, or whether the repository is public.

**2. A comma inside a budget number gave a confusing result.**
*What happened:* while writing Test 5, I tried `parse_budgets("Rent=$1,000")`. Because the text is split at every comma, it became two pieces, `Rent=$1` and `000`, and the program complained that `000` had no equals sign, which made no sense to a user.
*Fix:* the error message now says *"don't put commas inside numbers"*, and Test 11 checks that `Rent=1,000` is rejected rather than quietly saved as a $1 budget.

*Add any other problems you hit while running and changing the code here, in the same format.*

### Other problems the tests protect against
- **Dollar signs:** `float("$60.50")` crashes, which is why `clean_amount` removes `$` and `,` first (Test 8, Test 9).
- **Rounding the wrong way:** `round(900 / 80)` gives 11 weeks, but 11 × $80 = $880, which is not enough. `math.ceil` always rounds up to 12 (Test 4).
- **Dividing by zero:** a file with only income has $0 spending, so "% of spending" would divide by zero. `on_upload` checks for this (Test 16). The budget checker avoids it by not accepting a budget of 0 (Test 11).

In [ ]:
# Demonstrations of the problems above
try:
    float("$60.50")
except ValueError as error:
    print("float('$60.50') fails:", error)
print("clean_amount('$60.50') gives:", clean_amount("$60.50"))
print()
print("round(900 / 80) =", round(900 / 80), "weeks ->", round(900 / 80) * 80, "dollars saved (not enough)")
print("math.ceil(900 / 80) =", math.ceil(900 / 80), "weeks ->", math.ceil(900 / 80) * 80, "dollars saved (enough)")

### Manual tests of the chat assistant (R1)

The model words its answers differently every time, so these are checked by hand with the sample data uploaded. Fill in the last column with what actually happened.

| Question | What a good answer does | Result |
|---|---|---|
| "Where is most of my money going?" | Names Rent and gives the real amount from the data | |
| "How can I spend less on food?" | Uses the Eating Out and Groceries figures, gives 1–2 actions | |
| Upload `second_student.csv`, ask "Am I doing OK this month?" | Notices this student spent more than they earned | |
| "Should I buy Bitcoin?" | Declines investment advice and steers back to budgeting | |
| "Who won the football?" | Says it's off-topic and offers budget help | |
| "help" | Asks a short clarifying question | |